In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA
from langchain_classic.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter


c:\Users\samar\anaconda3\envs\rag_try\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
import os
import getpass
import dotenv

In [3]:
dotenv.load_dotenv()
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [6]:
file_path = "D:\\MadRocket\\rag_try\\rag_second_pdf\\PIIS0002934317309324.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=150)
texts = text_splitter.split_documents(documents)

print(f"Number of documents: {len(documents)}")
print(f"Number of text chunks: {len(texts)}")
print("Sample text chunk:")
print(texts[0])

Number of documents: 8
Number of text chunks: 78
Sample text chunk:
page_content='Headache
Paul Rizzoli, MD, FAHS,William J. Mullally, MD, FAHS
Graham Headache Center, Brigham and Women’s Faulkner Hospital, Harvard Medical School, Boston, Mass.
ABSTRACT
Headache, an almost universal human experience, is one of the most common complaints encountered in
medicine and neurology. Described and categorized since antiquity, with the ﬁrst classiﬁcation by Aretaeus
of Cappadocia, other classiﬁcations followed. The evaluation of this condition may be straightforward
or challenging, and, though often benign, headache may prove to be an ominous symptom. This review
discusses the current diagnosis and classiﬁcation of headache disorders and principles of management,' metadata={'producer': 'Acrobat Distiller 9.3.2 (Windows); modified using iText® 5.5.5 ©2000-2014 iText Group NV (AGPL-version)', 'creator': 'Elsevier', 'creationdate': '2017-12-06T20:00:33+00:00', 'author': 'Paul Rizzoli MD FAHS', 'cro

In [7]:
hf_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device":"cpu"})


In [8]:
vectorstore = FAISS.from_documents(texts, hf_embeddings)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [9]:
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.5)

In [10]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type="stuff")

In [11]:
def answer_question(question: str):
    result = qa_chain({"query": question})
    return result["result"], result["source_documents"]

question = "What are Risk factors associated with car racing?"
answer, sources = answer_question(question)
print(answer)
for idx, doc in enumerate(sources, start=1):
    print(f"\nSource {idx} metadata: {doc.metadata}")
    print(doc.page_content[:500])

C:\Users\samar\AppData\Local\Temp\ipykernel_16104\695500213.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = qa_chain({"query": question})


I don't know the answer to that question based on the provided context, which appears to be related to headaches and their diagnosis, rather than car racing.

Source 1 metadata: {'producer': 'Acrobat Distiller 9.3.2 (Windows); modified using iText® 5.5.5 ©2000-2014 iText Group NV (AGPL-version)', 'creator': 'Elsevier', 'creationdate': '2017-12-06T20:00:33+00:00', 'author': 'Paul Rizzoli MD FAHS', 'crossmarkdomains[1]': 'elsevier.com', 'crossmarkdomains[2]': 'sciencedirect.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'elsevierwebpdfspecifications': '6.5', 'keywords': '', 'moddate': '2017-12-07T12:19:42+08:00', 'subject': 'The American Journal of Medicine, 131 (2018) 17-24. doi:10.1016/j.amjmed.2017.09.005', 'title': 'Headache', 'doi': '10.1016/j.amjmed.2017.09.005', 'robots': 'noindex', 'source': 'D:\\MadRocket\\rag_try\\rag_second_pdf\\PIIS0002934317309324.pdf', 'total_pages': 8, 'page': 1, 'page_label': '18'}
 Age of onset
 Frequency, severity

In [13]:
question = "What are risk factor related to migraine?"
answer, sources = answer_question(question)
print(answer)
for idx, doc in enumerate(sources, start=1):
    print(f"\nSource {idx} metadata: {doc.metadata}")
    print(doc.page_content[:500])

According to the provided context, the risk factors associated with transformation to chronic migraine include:

1. Coexisting noncephalic sites of pain
2. Mood and anxiety disorders
3. Medication overuse
4. Obesity
5. Female sex
6. Lower educational status

Additionally, lifestyle features such as diet, caffeine use, sleep habits, work, and personal stress may also be relevant to migraine. Comorbid conditions like sleep disorders, depression, anxiety, and underlying medical disorders may also play a role.

Source 1 metadata: {'producer': 'Acrobat Distiller 9.3.2 (Windows); modified using iText® 5.5.5 ©2000-2014 iText Group NV (AGPL-version)', 'creator': 'Elsevier', 'creationdate': '2017-12-06T20:00:33+00:00', 'author': 'Paul Rizzoli MD FAHS', 'crossmarkdomains[1]': 'elsevier.com', 'crossmarkdomains[2]': 'sciencedirect.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'elsevierwebpdfspecifications': '6.5', 'keywords': '', 'moddate': '2017-12-07T12:19: